# SPHINX Injection Analysis

This notebook mirrors `Retrieval_analysis G-DAE.ipynb`, but it works on the SPHINX-injected observations stored in `../observations_sphinx/` while keeping the retrieval interpretation in PHOENIX.

# G-DAE reconstruction

In [ ]:
from __future__ import annotations

import os
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import matplotlib.pyplot as plt


# -----------------------------
# Paths
# -----------------------------
MODEL_PATH = Path("../Models/G-DAE.keras")
OBS_DIR = Path("../observations_sphinx")
BASELINE_OBS_DIR = Path("../observations")
CLEAN_PATH = Path("../pandexo_spec.txt")


# -----------------------------
# Reconstruction batch
# -----------------------------
RECONSTRUCTION_CASES: List[Dict[str, Path | str]] = [
    {
        "spec_name": "pandexo_output_10transits_fspot0.00_ffac0.00",
        "obs_dir": BASELINE_OBS_DIR,
        "save_dir": BASELINE_OBS_DIR,
    },
    {
        "spec_name": "pandexo_output_10transits_sphinx_fspot0.01_ffac0.08",
        "obs_dir": OBS_DIR,
        "save_dir": OBS_DIR,
    },
    {
        "spec_name": "pandexo_output_10transits_sphinx_fspot0.08_ffac0.54",
        "obs_dir": OBS_DIR,
        "save_dir": OBS_DIR,
    },
    {
        "spec_name": "pandexo_output_10transits_sphinx_fspot0.26_ffac0.70",
        "obs_dir": OBS_DIR,
        "save_dir": OBS_DIR,
    },
]



## Utilities

- Band-averaging of a high-resolution spectrum into instrument bins.
- Min–Max scaling for 1D traces (as used during AE training).
- Simple IO helpers for observation and clean reference spectra.


In [ ]:
def bin_average_with_halfbins(
    wl_src: np.ndarray,
    y_src: np.ndarray,
    centers: np.ndarray,
    halfwidths: np.ndarray,
    nsamp: int = 256,
) -> np.ndarray:
    """
    Band-average y_src over each band [c - h, c + h] using trapezoidal integration.

    Notes
    -----
    - The source wavelength grid is sorted internally.
    - np.interp applies edge clipping outside the source range.

    Parameters
    ----------
    wl_src
        Source wavelength grid (not necessarily sorted).
    y_src
        Source values evaluated at wl_src.
    centers
        Target bin centers.
    halfwidths
        Target bin half-widths (same length as centers).
    nsamp
        Number of samples used for numerical integration inside each bin.

    Returns
    -------
    np.ndarray
        Band-averaged values (len(centers),) in float32.
    """
    wl_src = np.asarray(wl_src, dtype=np.float64)
    y_src = np.asarray(y_src, dtype=np.float64)
    centers = np.asarray(centers, dtype=np.float64)
    halfwidths = np.asarray(halfwidths, dtype=np.float64)

    sort_idx = np.argsort(wl_src)
    wl_sorted = wl_src[sort_idx]
    y_sorted = y_src[sort_idx]

    out = np.empty_like(centers, dtype=np.float64)
    for i, (c, h) in enumerate(zip(centers, halfwidths)):
        a, b = c - h, c + h
        x = np.linspace(a, b, nsamp)
        yx = np.interp(x, wl_sorted, y_sorted)
        out[i] = np.trapz(yx, x) / (b - a)

    return out.astype(np.float32)


def normalize_min_max(arr: np.ndarray) -> np.ndarray:
    """
    Sample-wise Min–Max normalization for a 1D array.

    Returns zeros if the dynamic range is zero.
    """
    arr = np.asarray(arr, dtype=np.float32)
    min_val = float(arr.min())
    max_val = float(arr.max())
    rng = max_val - min_val

    if rng == 0.0:
        return np.zeros_like(arr)

    return (arr - min_val) / rng


def inverse_min_max_1d(normed: np.ndarray, ref: np.ndarray) -> np.ndarray:
    """
    Invert Min–Max normalization using the min/max of a reference array.
    """
    normed = np.asarray(normed, dtype=np.float32)
    ref = np.asarray(ref, dtype=np.float32)

    min_ref = float(ref.min())
    max_ref = float(ref.max())

    return normed * (max_ref - min_ref) + min_ref


def load_spectrum(spec_name: str, obs_dir: Path = OBS_DIR) -> np.ndarray:
    """
    Load an observation file as an (N, 4) array: (wl, Δwl, rp2/rs2, err).
    """
    path = obs_dir / f"{spec_name}.dat"
    if not path.is_file():
        raise FileNotFoundError(f"Observation not found: {path}")

    arr = np.loadtxt(path)
    if arr.ndim != 2 or arr.shape[1] < 4:
        raise ValueError(f"Expected (N, 4)+ columns, got shape {arr.shape}")

    return arr[:, :4].astype(np.float32)


def split_spectrum_columns(
    spectrum: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Split spectrum columns into typed vectors: (wl, Δwl, noisy, err).
    """
    wl = spectrum[:, 0].astype(np.float32)
    d_wl = spectrum[:, 1].astype(np.float32)
    y_noisy = spectrum[:, 2].astype(np.float32)
    y_err = spectrum[:, 3].astype(np.float32)
    return wl, d_wl, y_noisy, y_err


def load_clean_spectrum(clean_path: Path = CLEAN_PATH) -> Tuple[np.ndarray, np.ndarray]:
    """
    Load a clean spectrum with two columns: (wl, depth).
    """
    if not clean_path.is_file():
        raise FileNotFoundError(f"Clean spectrum not found: {clean_path}")

    arr = np.loadtxt(clean_path)
    if arr.ndim != 2 or arr.shape[1] < 2:
        raise ValueError(f"Expected at least 2 columns, got shape {arr.shape}")

    wl_clean = arr[:, 0].astype(np.float32)
    y_clean = arr[:, 1].astype(np.float32)
    return wl_clean, y_clean


def force_reverse_order(
    wl: np.ndarray,
    d_wl: np.ndarray,
    y_noisy: np.ndarray,
    y_err: np.ndarray,
    y_clean_binned: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Reverse all vectors along axis 0. Use only if your files are known to be descending.
    """
    return (
        wl[::-1].copy(),
        d_wl[::-1].copy(),
        y_noisy[::-1].copy(),
        y_err[::-1].copy(),
        y_clean_binned[::-1].copy(),
    )


## Autoencoder helpers

- Loads the trained Keras model from disk.
- Applies min–max scaling to the noisy trace.
- Runs inference and rescales output back to the clean reference range.

In [ ]:
def load_autoencoder(model_path: Path = MODEL_PATH):
    """
    Load a Keras autoencoder from disk.

    Import is local to avoid requiring TensorFlow unless reconstruction is used.
    """
    from tensorflow import keras  # local import by design

    if not model_path.is_file():
        raise FileNotFoundError(f"Model not found: {model_path}")

    return keras.models.load_model(model_path)


def reconstruct_trace(
    autoencoder,
    y_noisy: np.ndarray,
    clean_ref: np.ndarray,
) -> np.ndarray:
    """
    Reconstruct a 1D trace with a trained autoencoder.
    """
    n_features = int(autoencoder.input_shape[-1])
    if len(y_noisy) != n_features or len(clean_ref) != n_features:
        raise ValueError(
            "Input length mismatch: "
            f"model expects {n_features}, got noisy={len(y_noisy)}, clean={len(clean_ref)}."
        )

    x_norm = normalize_min_max(y_noisy)
    x_in = x_norm.reshape(1, -1).astype(np.float32)

    y_pred_norm = autoencoder.predict(x_in, verbose=0)[0].astype(np.float32)
    y_recon = inverse_min_max_1d(y_pred_norm, clean_ref)

    return y_recon.astype(np.float32)


def save_reconstructed(
    spec_name: str,
    wl: np.ndarray,
    d_wl: np.ndarray,
    y_recon: np.ndarray,
    y_err: np.ndarray,
    obs_dir: Path = OBS_DIR,
) -> Path:
    """
    Save a 4-column file (wl, Δwl, recon, err) inside obs_dir.
    """
    obs_dir.mkdir(parents=True, exist_ok=True)
    out = np.column_stack((wl, d_wl, y_recon, y_err)).astype(np.float64)

    out_path = obs_dir / f"{spec_name}_recon.dat"
    np.savetxt(out_path, out, fmt="%.10e")
    return out_path


def plot_quicklook(
    wl: np.ndarray,
    y_noisy: np.ndarray,
    y_recon: np.ndarray,
    y_clean: np.ndarray,
) -> None:
    """
    Quick-look plot comparing noisy input, AE reconstruction, and clean reference.
    """
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(wl, y_noisy, label="Noisy input")
    ax.plot(wl, y_recon, label="Reconstructed (AE)")
    ax.plot(wl, y_clean, "--", label="Clean reference")

    ax.set_xlabel("Wavelength (um)")
    ax.set_ylabel("Transit depth")
    ax.legend()

    fig.tight_layout()
    plt.show()


## Decontammination

In [ ]:
def reconstruct_and_save(
    spec_name: str,
    model_path: Path = MODEL_PATH,
    obs_dir: Path = OBS_DIR,
    save_dir: Path | None = None,
    clean_path: Path = CLEAN_PATH,
    nsamp: int = 256,
    show_plot: bool = False,
    force_flip: bool = True,
) -> Path:
    """
    End-to-end pipeline: load inputs, bin clean spectrum, run AE reconstruction,
    and save a 4-column file with (wl, Δwl, recon, err).
    """
    if save_dir is None:
        save_dir = obs_dir

    spectrum = load_spectrum(spec_name, obs_dir=obs_dir)
    wl, d_wl, y_noisy, y_err = split_spectrum_columns(spectrum)

    wl_clean, y_clean = load_clean_spectrum(clean_path=clean_path)

    y_clean_binned = bin_average_with_halfbins(
        wl_src=wl_clean,
        y_src=y_clean,
        centers=wl,
        halfwidths=d_wl,
        nsamp=nsamp,
    )

    if force_flip:
        wl, d_wl, y_noisy, y_err, y_clean_binned = force_reverse_order(
            wl, d_wl, y_noisy, y_err, y_clean_binned
        )

    autoencoder = load_autoencoder(model_path=model_path)
    y_recon = reconstruct_trace(
        autoencoder=autoencoder,
        y_noisy=y_noisy,
        clean_ref=y_clean_binned,
    )

    out_path = save_reconstructed(
        spec_name=spec_name,
        wl=wl,
        d_wl=d_wl,
        y_recon=y_recon,
        y_err=y_err,
        obs_dir=save_dir,
    )

    if show_plot:
        plot_quicklook(wl=wl, y_noisy=y_noisy, y_recon=y_recon, y_clean=y_clean_binned)

    return out_path


recon_paths: List[Path] = []
for case in RECONSTRUCTION_CASES:
    path_out = reconstruct_and_save(
        spec_name=str(case["spec_name"]),
        model_path=MODEL_PATH,
        obs_dir=Path(case["obs_dir"]),
        save_dir=Path(case["save_dir"]),
        clean_path=CLEAN_PATH,
        nsamp=256,
        show_plot=True,
        force_flip=True,
    )
    recon_paths.append(path_out)
    print(f"Saved: {path_out}")


# POSEIDON retrieval (single dataset)

This section:
1) Defines the star and planet objects.
2) Loads a reconstructed observation file and initializes instrument properties.
3) Defines the retrieval model + priors.
4) Reads opacities and produces standard retrieval plots (spectrum + corner).
5) Computes MSE and reduced χ² against the clean truth rebinned to the dataset bins,
   and appends a row to a CSV log file.


In [ ]:
import numpy as np

from POSEIDON.core import create_star, create_planet
from POSEIDON.constants import R_Sun, R_E, M_E


# -----------------------------
# Star definition (TRAPPIST-1)
# -----------------------------
R_s = 0.1192 * R_Sun
T_s = 2566.0
met_s = 0.00
log_g_s = 5.2396

star = create_star(R_s, T_s, log_g_s, met_s, stellar_grid="phoenix")


# -----------------------------
# Planet definition (TRAPPIST-1e)
# -----------------------------
planet_name = "Trappist-1e"
R_p = 0.917985 * R_E
M_p = 0.6356 * M_E
T_eq = 255.0

planet = create_planet(planet_name, R_p, mass=M_p, T_eq=T_eq)


In [ ]:
from POSEIDON.core import load_data, wl_grid_constant_R
from POSEIDON.visuals import plot_data


observation_file = "pandexo_output_10transits_sphinx_fspot0.26_ffac0.70_recon.dat"
data_dir = "../observations_sphinx"
datasets = [observation_file]
instruments = ["JWST_NIRSpec_PRISM"]

wl_min = 0.4
wl_max = 6.0
R_grid = 4000

wl_model = wl_grid_constant_R(wl_min, wl_max, R_grid)
data = load_data(data_dir, datasets, instruments, wl_model)

fig_data = plot_data(data, planet_name)

In [ ]:
from POSEIDON.core import define_model, set_priors


model_name = "recon_10T_0.00spot-0.00fac"
bulk_species = ["N2"]
param_species = ["H2O", "CH4", "CO2", "O3"]

model = define_model(
    model_name,
    bulk_species,
    param_species,
    PT_profile="isotherm",
    cloud_model="cloud-free",
)

print("Free parameters:", model["param_names"])

prior_types = {
    "T": "uniform",
    "R_p_ref": "uniform",
    "log_H2O": "uniform",
    "log_CH4": "uniform",
    "log_CO2": "uniform",
    "log_O3": "uniform",
}

prior_ranges = {
    "T": [200, 400],
    "R_p_ref": [0.85 * R_p, 1.15 * R_p],
    "log_H2O": [-8, -1],
    "log_CH4": [-8, -1],
    "log_CO2": [-5, -1],
    "log_O3": [-8, -1],
}

priors = set_priors(planet, star, model, data, prior_types, prior_ranges)

In [ ]:
from POSEIDON.core import read_opacities


opacity_treatment = "opacity_sampling"

T_fine = np.arange(200, 401, 10)
log_P_fine = np.arange(-2, 2.0001, 0.2)

opac = read_opacities(model, wl_model, opacity_treatment, T_fine, log_P_fine)

In [ ]:
from POSEIDON.utility import read_retrieved_spectrum, plot_collection
from POSEIDON.visuals import plot_spectra_retrieved
from POSEIDON.corner import generate_cornerplot


wl_out, spec_low2, spec_low1, spec_median, spec_high1, spec_high2 = read_retrieved_spectrum(
    planet_name, model_name
)

spectra_median = plot_collection(spec_median, wl_out, collection=[])
spectra_low1 = plot_collection(spec_low1, wl_out, collection=[])
spectra_low2 = plot_collection(spec_low2, wl_out, collection=[])
spectra_high1 = plot_collection(spec_high1, wl_out, collection=[])
spectra_high2 = plot_collection(spec_high2, wl_out, collection=[])

fig_spec = plot_spectra_retrieved(
    spectra_median,
    spectra_low2,
    spectra_low1,
    spectra_high1,
    spectra_high2,
    planet_name,
    data,
    R_to_bin=100,
    data_labels=["NIRSpec_PRISM"],
    data_colour_list=["lime"],
)

fig_corner = generate_cornerplot(planet, model)


## Metrics vs clean truth (binned to the dataset)

We compute:
- MSE between median retrieved spectrum (binned to the dataset) and the clean truth (binned).
- Reduced χ² using observational errors.

Important:
- `n_free_params` must match the retrieval free-parameter count for a meaningful reduced χ².



In [ ]:
from pathlib import Path

from POSEIDON.instrument import bin_spectrum_to_data
from POSEIDON.utility import read_data


n_free_params = 11
log_path = Path("chi2_log_sphinx_gdae.csv")

# Load clean truth
wl_clean, y_clean = load_clean_spectrum(clean_path=CLEAN_PATH)

# Observed dataset grid + half-bin widths + errors
wl_data, half_bin, y_obs, err_obs = read_data(data_dir, observation_file)

# Rebin retrieved median spectrum to dataset bins
model_binned = bin_spectrum_to_data(spec_median, wl_out, data)

# Rebin clean truth to dataset bins (band-average)
clean_binned = bin_average_with_halfbins(wl_clean, y_clean, wl_data, half_bin, nsamp=256)

if not (len(model_binned) == len(clean_binned) == len(err_obs)):
    raise ValueError("Binned arrays and errors must have the same length.")

sig = np.asarray(err_obs, dtype=float)
if np.any(sig <= 0):
    raise ValueError("Found non-positive uncertainties in err_obs; χ² is not valid.")

resid = np.asarray(model_binned, dtype=float) - np.asarray(clean_binned, dtype=float)

N = int(resid.size)
p = int(n_free_params)
dof = int(max(N - p, 0))

chi2 = float(np.sum((resid / sig) ** 2))
chi2_reduced = chi2 / dof if dof > 0 else np.nan

mse = float(np.mean(resid**2))
rmse = float(np.sqrt(mse))
rmse_ppm = 1e6 * rmse

print("---- Metrics vs clean truth (no mask) ----")
print(f"N points      : {N}")
print(f"Free params p : {p}")
print(f"DoF           : {dof}")
print(f"MSE           : {mse:.6e}")
print(f"RMSE          : {rmse:.6e} ({rmse_ppm:.2f} ppm)")
print(f"chi2          : {chi2:.6f}")
print(f"chi2_reduced  : {chi2_reduced:.6f}")

row = {
    "planet_name": planet_name,
    "model_name": model_name,
    "N": N,
    "p": p,
    "dof": dof,
    "MSE": mse,
    "chi2": chi2,
    "chi2_reduced": chi2_reduced,
}

if log_path.exists():
    df_log = pd.read_csv(log_path)
    df_log = pd.concat([df_log, pd.DataFrame([row])], ignore_index=True)
else:
    df_log = pd.DataFrame([row])

df_log.to_csv(log_path, index=False, float_format="%.10g")
print(f"Appended row to: {log_path.resolve()}")
